In [1]:
import triton
import torch
import math
from tabulate import tabulate
import triton.language as tl
import os
os.environ["TRITON_PRINT_AUTOTUNING"] = "1"

@triton.autotune(
    configs=[
        triton.Config({'BLOCK_Q': 64, 'BLOCK_K': 64}, num_stages=2, num_warps=4),
        triton.Config({'BLOCK_Q': 128, 'BLOCK_K': 64}, num_stages=3, num_warps=8),
        triton.Config({'BLOCK_Q': 64, 'BLOCK_K': 128}, num_stages=3, num_warps=8),
    ],
    key=['N_q', 'N_k', 'd'],
)
@triton.jit
def triton_flash_attn2_kernel(
    Q_ptr, K_ptr, V_ptr, O_ptr,
    stride_qm, stride_qd,
    stride_kd, stride_kn,
    stride_vn, stride_vd,
    stride_om, stride_od,
    N_q, N_k,
    sm_scale,
    BLOCK_Q: tl.constexpr, BLOCK_K: tl.constexpr,
    d: tl.constexpr
):
    pid_q = tl.program_id(axis=0)
    
    offs_q = pid_q * BLOCK_Q + tl.arange(0, BLOCK_Q)
    offs_d = tl.arange(0, d)
    
    # Load Q into registers
    q_ptrs = Q_ptr + offs_q[:, None] * stride_qm + offs_d[None, :] * stride_qd
    q = tl.load(q_ptrs, mask=offs_q[:, None] < N_q, other=0.0)
    
    m_i = tl.full([BLOCK_Q, 1], value=float("-inf"), dtype=tl.float32)
    l_i = tl.zeros([BLOCK_Q, 1], dtype=tl.float32)
    acc = tl.zeros([BLOCK_Q, d], dtype=tl.float32)
    
    for start_k in range(0, N_k, BLOCK_K):
        offs_k = start_k + tl.arange(0, BLOCK_K)
        
        
        # Pointer arithmetic
        k_ptrs = K_ptr + offs_d[:, None] * stride_kd + offs_k[None, :] * stride_kn
        v_ptrs = V_ptr + offs_k[:, None] * stride_vn + offs_d[None, :] * stride_vd

        # Load into shared memory via blocking
        k = tl.load(k_ptrs, mask=(offs_d[:, None] < d) & (offs_k[None, :] < N_k), other=0.0)
        v = tl.load(v_ptrs, mask=(offs_k[:, None] < N_k) & (offs_d[None, :] < d), other=0.0)
        
        # Compute logits
        s = tl.dot(q, k, allow_tf32=True) * sm_scale
        s = tl.where(offs_k[None, :] < N_k, s, float("-inf"))
        
        # Online Rescaling
        m_ij = tl.max(s, axis=1)[:, None]
        m_next = tl.maximum(m_i, m_ij)
        alpha = tl.exp(m_i - m_next)
        p_tilde = tl.exp(s - m_next)
        
        # Accumulate in registers
        acc = acc * alpha
        acc += tl.dot(p_tilde.to(tl.float16), v.to(tl.float16), allow_tf32=True).to(tl.float32)
        
        l_i = l_i * alpha + tl.sum(p_tilde, axis=1)[:, None]
        m_i = m_next
        
    O = acc / (l_i + 1e-6)
    
    o_ptrs = O_ptr + offs_q[:, None] * stride_om + offs_d[None, :] * stride_od
    tl.store(o_ptrs, O, mask=offs_q[:, None] < N_q)

In [4]:
def triton_flash_attention2(q, k, v, sm_scale=None):
    N_q, d = q.shape
    k_t = k.T.contiguous()
    _, N_k = k_t.shape  # k_t layout geometry is [d, N_k]
    if sm_scale is None:
        sm_scale = 1.0 / math.sqrt(d)
        
    O = torch.empty_like(q)
    
    grid = lambda meta: (triton.cdiv(N_q, meta['BLOCK_Q']),)
    
    triton_flash_attn2_kernel[grid](
        q, k_t, v, O,
        q.stride(0), q.stride(1),
        k_t.stride(0), k_t.stride(1),  # k_t.stride(0) is stride_kd, k_t.stride(1) is stride_kn
        v.stride(0), v.stride(1),
        O.stride(0), O.stride(1),
        N_q=N_q, N_k=N_k, sm_scale=sm_scale,
        d=d
    )
    return O


In [5]:
torch.manual_seed(97)

N_q, N_k, d = 512, 512, 64
sm_scale = 1.0 / math.sqrt(d)

q = torch.randn((N_q, d), dtype=torch.float16, device="cuda")
k = torch.randn((N_k, d), dtype=torch.float16, device="cuda")
v = torch.randn((N_k, d), dtype=torch.float16, device="cuda")

print("Computing PyTorch Native SDPA Reference (FP16)...")
from torch.nn.attention import sdpa_kernel, SDPBackend
with sdpa_kernel([SDPBackend.FLASH_ATTENTION, SDPBackend.EFFICIENT_ATTENTION]):
    out_torch = torch.nn.functional.scaled_dot_product_attention(
        q.unsqueeze(0).unsqueeze(0), 
        k.unsqueeze(0).unsqueeze(0), 
        v.unsqueeze(0).unsqueeze(0), 
        scale=sm_scale
    ).squeeze(0).squeeze(0)
    
print("Computing Custom Optimized Triton Flash-Attention (FP16)...")
out_triton = triton_flash_attention2(q, k, v, sm_scale=sm_scale)

atol = 1e-3
rtol = 1e-3
max_diff = torch.max(torch.abs(out_torch - out_triton)).item()
is_close = torch.allclose(out_torch, out_triton, atol=atol, rtol=rtol)

print("\n" + "="*40)
print("         FP16 VERIFICATION RESULTS")
print("="*40)
print(f"Max Absolute Value Discrepancy: {max_diff:.2e}")
print(f"Validation Tolerance Threshold:  {atol:.2e}")
print("-"*40)

if is_close:
    print("SUCCESS: The optimized Triton kernel matches PyTorch perfectly!")
else:
    print("FAILURE: Discrepancy detected.")

Computing PyTorch Native SDPA Reference (FP16)...
Computing Custom Optimized Triton Flash-Attention (FP16)...
Autotuning kernel triton_flash_attn2_kernel with config BLOCK_Q: 64, BLOCK_K: 64, num_warps: 4, num_ctas: 1, num_stages: 2, maxnreg: None
Autotuning kernel triton_flash_attn2_kernel with config BLOCK_Q: 128, BLOCK_K: 64, num_warps: 8, num_ctas: 1, num_stages: 3, maxnreg: None
Autotuning kernel triton_flash_attn2_kernel with config BLOCK_Q: 64, BLOCK_K: 128, num_warps: 8, num_ctas: 1, num_stages: 3, maxnreg: None
Triton autotuning for function triton_flash_attn2_kernel,
with key as (512, 512, 64, 'torch.float16', 'torch.float16', 'torch.float16', 'torch.float16'),
finished after 5.21s,
best config selected: BLOCK_Q: 64, BLOCK_K: 128, num_warps: 8, num_ctas: 1, num_stages: 3, maxnreg: None;

         FP16 VERIFICATION RESULTS
Max Absolute Value Discrepancy: 2.44e-04
Validation Tolerance Threshold:  1.00e-03
----------------------------------------
SUCCESS: The optimized Triton ke

In [6]:
def naive_attention(q, k, v, sm_scale):
    attn_weights = torch.matmul(q, k.T) * sm_scale
    attn_probs = torch.softmax(attn_weights, dim=-1)
    return torch.matmul(attn_probs, v)



d = 64
sequence_lengths = [1024, 2048, 4096, 8192, 16384]
results_table = []

print("Running comprehensive multi-backend FP16 benchmarks...")

for N in sequence_lengths:
   
    q = torch.randn((N, d), dtype=torch.float16, device="cuda")
    k = torch.randn((N, d), dtype=torch.float16, device="cuda")
    v = torch.randn((N, d), dtype=torch.float16, device="cuda")
    sm_scale = 1.0 / math.sqrt(d)
    
    
    
    # Warmup passes for all engines to clear initialization overhead
    _ = naive_attention(q, k, v, sm_scale)
    _ = triton_flash_attention2(q, k, v, sm_scale)  
    start_event = torch.cuda.Event(enable_timing=True)
    end_event = torch.cuda.Event(enable_timing=True)


    
    # Profile Naive PyTorch Attention (Eager Mode)
    start_event.record()
    for _ in range(20):
        _ = naive_attention(q, k, v, sm_scale)
    end_event.record()
    torch.cuda.synchronize()
    naive_time = start_event.elapsed_time(end_event) / 20

    
    
    # Profile PyTorch Native (SDPA Flash)
    start_event.record()
    with sdpa_kernel([SDPBackend.FLASH_ATTENTION]):
        for _ in range(20):
            _ = torch.nn.functional.scaled_dot_product_attention(
                q.unsqueeze(0).unsqueeze(0), k.unsqueeze(0).unsqueeze(0), v.unsqueeze(0).unsqueeze(0), scale=sm_scale
            )
    end_event.record()
    torch.cuda.synchronize()
    torch_opt_time = start_event.elapsed_time(end_event) / 20


    
    # Profile Your Custom Triton Kernel
    start_event.record()
    for _ in range(20):
        _ = triton_flash_attention2(q, k, v, sm_scale)
        
    end_event.record()
    torch.cuda.synchronize()
    triton_time = start_event.elapsed_time(end_event) / 20
    
    # Calculate Speedup over the Naive implementation
    naive_speedup = naive_time / triton_time
    sdpa_speedup = torch_opt_time / triton_time
    
    results_table.append([
        N, 
        f"{naive_time:.3f} ms", 
        f"{torch_opt_time:.3f} ms", 
        f"{triton_time:.3f} ms", 
        f"{naive_speedup:.2f}x",
        f"{sdpa_speedup:.2f}x"
    ])
    
print("\n" + "="*85)
print(f"               COMPREHENSIVE ATTENTION BENCHMARK PROFILE (FP16)")
print("="*85)
headers = ["Seq Length (N)", "Naive PyTorch", "PyTorch SDPA", "Custom Triton", "vs Naive", "vs SDPA"]
print(tabulate(results_table, headers=headers, tablefmt="grid"))

Running comprehensive multi-backend FP16 benchmarks...
Autotuning kernel triton_flash_attn2_kernel with config BLOCK_Q: 64, BLOCK_K: 64, num_warps: 4, num_ctas: 1, num_stages: 2, maxnreg: None
Autotuning kernel triton_flash_attn2_kernel with config BLOCK_Q: 128, BLOCK_K: 64, num_warps: 8, num_ctas: 1, num_stages: 3, maxnreg: None
Autotuning kernel triton_flash_attn2_kernel with config BLOCK_Q: 64, BLOCK_K: 128, num_warps: 8, num_ctas: 1, num_stages: 3, maxnreg: None
Triton autotuning for function triton_flash_attn2_kernel,
with key as (1024, 1024, 64, 'torch.float16', 'torch.float16', 'torch.float16', 'torch.float16'),
finished after 0.37s,
best config selected: BLOCK_Q: 64, BLOCK_K: 128, num_warps: 8, num_ctas: 1, num_stages: 3, maxnreg: None;
Autotuning kernel triton_flash_attn2_kernel with config BLOCK_Q: 64, BLOCK_K: 64, num_warps: 4, num_ctas: 1, num_stages: 2, maxnreg: None
Autotuning kernel triton_flash_attn2_kernel with config BLOCK_Q: 128, BLOCK_K: 64, num_warps: 8, num_ctas: 